# Membership inference on a DDPM: members come back unchanged

One **SDEdit round-trip** as a membership-inference attack: add noise to an
image up to a fraction `t_edit` of the diffusion chain, let the model denoise
it back (DDIM), and score the image by `pixel-MSE(x0, roundtrip(x0))`. A small
DDPM (`diffusers.UNet2DModel`) is trained **from scratch in this notebook** on
`N_MEMBER` CIFAR-10 images until it overfits; images it was trained on come
back almost unchanged, images it never saw get pulled somewhere else.

Evaluation follows **Carlini, Chien, Nasr, Song, Terzis, Tramèr,
"Membership Inference Attacks From First Principles," IEEE S&P 2022**: ROC on
log-log axes and TPR at a low FPR (1 %, 5 %) instead of accuracy or AUROC
alone. The paper is cited for the *evaluation convention*; the attack and its
numbers are the lecturer's own.

**Runtime.** Open in Google Colab and switch to a GPU first:
*Runtime → Change runtime type → T4 GPU*. The default configuration
(n_member = 500) takes roughly 9–15 minutes on a free T4, most of it
training. It also runs on Apple-silicon (MPS) or CPU, just slower — see the
`TRAIN_STEPS` note in the config cell. The optional larger-n comparison at
the end is **off by default** (it roughly doubles the runtime).

**These numbers are not the slide's numbers.** The figures on the lecture
slide (AUROC ≈ 0.996 at n_member = 500, falling to ≈ 0.65 at n_member = 10 000)
come from the lecturer's own, much longer research run (400 epochs per
model, not included in this repository). This notebook reproduces the
*method* in a Colab-sized budget: expect a clear membership signal at
n_member = 500, but a lower AUROC than the slide, and slightly different
numbers from run to run.

## Setup

Colab already ships `torch`, `torchvision`, `scikit-learn` and `matplotlib`;
`diffusers` is installed on first run if it is missing. Locally:
`pip install torch torchvision diffusers scikit-learn matplotlib`.

In [ ]:
import importlib.util, subprocess, sys

if importlib.util.find_spec("diffusers") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "diffusers"])

import json, math, os, time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import torchvision
from torchvision import transforms
from torchvision.utils import make_grid
from diffusers import DDIMScheduler, DDPMScheduler, UNet2DModel
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt

TEAL, SLATE, RED = "#0f766e", "#475569", "#dc2626"

In [ ]:
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print("device:", DEVICE, f"({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else "")
if DEVICE == "cpu":
    print("No GPU found. In Colab: Runtime -> Change runtime type -> T4 GPU, then re-run. "
          "On CPU, training the default config takes hours; lower TRAIN_STEPS to try the code.")

## Config

`TRAIN_STEPS` is the whole training budget (optimizer steps, batch 64).
At n_member = 500 that is `TRAIN_STEPS / 8` epochs — the model needs to see
each image many hundreds of times before it memorizes it, and the membership
signal grows with training (see the exercises). The environment variables
only exist so the notebook can be smoke-tested with a smaller budget; in
Colab just edit the numbers.

In [ ]:
N_MEMBER = 500        # training-set size of the target DDPM (the memorization gate)
PER_GROUP = 500       # images scored per group (members / non-members)
TRAIN_STEPS = int(os.environ.get("MIA_TRAIN_STEPS", 5000))
BATCH = 64
LR = 1e-3
T_EDIT = 0.25         # fraction of the diffusion chain the round-trip noises to
DDIM_STEPS = 25       # DDIM schedule over the full chain; ~T_EDIT * 25 of them are run
NUM_TRAIN_TIMESTEPS = 1000
SEED = 0
TEST_SEED = 1234      # non-member permutation of the CIFAR-10 test set
RUN_LARGE = os.environ.get("MIA_RUN_LARGE", "0") == "1"   # optional larger-n cell at the end
N_LARGE = 5000
DATA_ROOT = "data"
OUT = Path("out"); OUT.mkdir(exist_ok=True)
torch.manual_seed(SEED)

## Data: CIFAR-10 and the member / non-member split

CIFAR-10 (Krizhevsky, 2009) is downloaded by torchvision (~170 MB, usually
under a minute in Colab). Members are the first `N_MEMBER` images of the
**train** set under `torch.randperm(…, generator=manual_seed(0))`;
non-members are the first `PER_GROUP` images of the **test** set under
`manual_seed(1234)` — the same split convention as the lecturer's research
code, so a larger `N_MEMBER` always contains the smaller member sets. Pixels
are scaled to [-1, 1]; no augmentation (flips would make memorization
harder and the gate less visible).

In [ ]:
tfm = transforms.Compose([transforms.ToTensor(), transforms.Normalize([0.5] * 3, [0.5] * 3)])
ds_train = torchvision.datasets.CIFAR10(DATA_ROOT, train=True, download=True, transform=tfm)
ds_test = torchvision.datasets.CIFAR10(DATA_ROOT, train=False, download=True, transform=tfm)


def member_indices(n_member):
    g = torch.Generator().manual_seed(0)
    return torch.randperm(len(ds_train), generator=g)[:n_member].tolist()


g_te = torch.Generator().manual_seed(TEST_SEED)
nonmember_idx = torch.randperm(len(ds_test), generator=g_te)[:PER_GROUP].tolist()
nonmember_x = torch.stack([ds_test[i][0] for i in nonmember_idx])
print("non-members:", tuple(nonmember_x.shape))

## Model and training

A small `UNet2DModel` (~4 M parameters; one self-attention block at 8×8)
trained with the standard DDPM ε-prediction loss and a linear β schedule
over 1000 timesteps. On CUDA it uses fp16 autocast, which is what makes a T4
fast enough for this budget.

In [ ]:
def make_unet():
    return UNet2DModel(
        sample_size=32, in_channels=3, out_channels=3, layers_per_block=1,
        block_out_channels=(64, 128, 128),
        down_block_types=("DownBlock2D", "DownBlock2D", "AttnDownBlock2D"),
        up_block_types=("AttnUpBlock2D", "UpBlock2D", "UpBlock2D"),
    )


ddpm = DDPMScheduler(num_train_timesteps=NUM_TRAIN_TIMESTEPS, beta_schedule="linear")
ddim = DDIMScheduler(num_train_timesteps=NUM_TRAIN_TIMESTEPS, beta_schedule="linear")


def train_ddpm(x_train, steps, seed=SEED, log_every=500):
    torch.manual_seed(seed)
    unet = make_unet().to(DEVICE).train()
    opt = torch.optim.AdamW(unet.parameters(), lr=LR, weight_decay=0.0)
    warmup = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / 100))
    amp = DEVICE == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    x_train = x_train.to(DEVICE)
    g = torch.Generator().manual_seed(seed)
    step, t0 = 0, time.time()
    while step < steps:
        for b in torch.randperm(len(x_train), generator=g).split(BATCH):
            x0 = x_train[b.to(DEVICE)]
            noise = torch.randn_like(x0)
            t = torch.randint(0, NUM_TRAIN_TIMESTEPS, (len(x0),), device=DEVICE)
            with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
                loss = F.mse_loss(unet(ddpm.add_noise(x0, noise, t), t).sample.float(), noise)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); warmup.step()
            step += 1
            if step % log_every == 0 or step == steps:
                print(f"step {step}/{steps}  loss {loss.item():.4f}  {time.time() - t0:.0f}s", flush=True)
            if step >= steps:
                break
    return unet.eval()

In [ ]:
member_idx = member_indices(N_MEMBER)
member_train = torch.stack([ds_train[i][0] for i in member_idx])
member_x = member_train[:PER_GROUP]
print(f"training on n_member={N_MEMBER}: {TRAIN_STEPS} steps = "
      f"{TRAIN_STEPS * BATCH / N_MEMBER:.0f} epochs")
t_train = time.time()
unet = train_ddpm(member_train, TRAIN_STEPS)
print(f"trained in {time.time() - t_train:.0f}s")

## The attack: one round-trip, pixel-MSE score

The whole attack is the next cell. `roundtrip` noises `x0` to the DDIM
timestep closest to `T_EDIT · 1000` with a fixed noise seed and denoises it
back with the deterministic DDIM sampler. The score is the per-image
pixel-MSE between input and output; **low MSE ⇒ "member"**.

`tpr_at` reads the ROC as a step function — the best TPR among thresholds
whose FPR is at most the target — rather than interpolating between points.

In [ ]:
@torch.no_grad()
def roundtrip(unet, x0, t_edit=T_EDIT, seed=0, batch=250):
    ddim.set_timesteps(DDIM_STEPS)
    ts = [int(t) for t in ddim.timesteps if t <= t_edit * NUM_TRAIN_TIMESTEPS]
    g = torch.Generator().manual_seed(seed)
    out = []
    for xb in x0.split(batch):
        noise = torch.randn(xb.shape, generator=g)
        x = ddim.add_noise(xb, noise, torch.tensor([ts[0]])).to(DEVICE)
        for t in ts:
            x = ddim.step(unet(x, t).sample, t, x).prev_sample
        out.append(x.cpu())
    return torch.cat(out)


def mse(a, b):
    return ((a - b) ** 2).flatten(1).mean(1).numpy()


def tpr_at(fpr, tpr, max_fpr):
    return float(tpr[fpr <= max_fpr].max())


def evaluate(mse_member, mse_nonmember):
    y = np.r_[np.ones(len(mse_member)), np.zeros(len(mse_nonmember))]
    score = -np.r_[mse_member, mse_nonmember]          # low MSE = member
    fpr, tpr, _ = roc_curve(y, score)
    return {"auroc": float(roc_auc_score(y, score)),
            "tpr_at_1pct": tpr_at(fpr, tpr, 0.01), "tpr_at_5pct": tpr_at(fpr, tpr, 0.05),
            "mse_member_mean": float(mse_member.mean()),
            "mse_nonmember_mean": float(mse_nonmember.mean()), "fpr": fpr, "tpr": tpr}


def attack(unet, member_x, label):
    m1 = roundtrip(unet, member_x, seed=0)
    n1 = roundtrip(unet, nonmember_x, seed=1)
    mse_m, mse_n = mse(m1, member_x), mse(n1, nonmember_x)
    r = evaluate(mse_m, mse_n)
    print(f"{label}: AUROC={r['auroc']:.3f}  TPR@1%FPR={r['tpr_at_1pct']:.3f}  "
          f"TPR@5%FPR={r['tpr_at_5pct']:.3f}  mean MSE member={r['mse_member_mean']:.4f} "
          f"non-member={r['mse_nonmember_mean']:.4f}")
    return r, m1, n1, mse_m, mse_n


res_small, member_rt, nonmember_rt, mse_m, mse_n = attack(unet, member_x, f"n_member={N_MEMBER}")

## ROC on log-log axes

Random guessing is the diagonal. What matters for privacy is the far left of
the plot: how many members the attack identifies while almost never
accusing a non-member (TPR at 1 % FPR, marked by the dot). With 500
non-members, the smallest non-zero FPR is 0.002, so the curve starts there.

In [ ]:
def plot_roc_loglog(curves, path=None):
    fig, ax = plt.subplots(figsize=(6, 4.2), dpi=100)
    colors = [TEAL, RED, SLATE]
    for (label, r), color in zip(curves.items(), colors):
        ax.plot(np.clip(r["fpr"], 1e-3, 1), np.clip(r["tpr"], 1e-3, 1), color=color, linewidth=2.2,
                label=f"{label} (AUROC={r['auroc']:.3f})")
        ax.scatter([0.01], [max(r["tpr_at_1pct"], 1e-3)], color=color, zorder=5, s=35)
    ax.plot([1e-3, 1], [1e-3, 1], color=SLATE, linestyle="--", linewidth=1, alpha=0.5)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlim(1e-3, 1); ax.set_ylim(1e-3, 1.05)
    ax.set_xlabel("False positive rate", fontsize=13)
    ax.set_ylabel("True positive rate", fontsize=13)
    ax.set_title("Membership inference ROC (log-log)", fontsize=13)
    ax.legend(fontsize=10, loc="lower right", frameon=False)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    fig.tight_layout()
    if path:
        fig.savefig(path, facecolor="white")
    plt.show()


plot_roc_loglog({f"n={N_MEMBER}": res_small}, OUT / "roc_loglog.png")

## What the round-trip does to an image

Top two rows: the first eight members before and after one round-trip.
Bottom two rows: the first eight non-members. The number above each pair is
its pixel-MSE. Members come back (almost) unchanged; non-members come back
as a plausible-looking but *different* image.

In [ ]:
def show_pairs(x_before, x_after, scores, title, path=None, n=8):
    fig, axes = plt.subplots(2, n, figsize=(n * 1.3, 3.0), dpi=100)
    for j in range(n):
        for row, x in enumerate((x_before, x_after)):
            ax = axes[row, j]
            ax.imshow((x[j].permute(1, 2, 0) * 0.5 + 0.5).clamp(0, 1).numpy())
            ax.set_xticks([]); ax.set_yticks([])
        axes[0, j].set_title(f"{scores[j]:.4f}", fontsize=9)
    axes[0, 0].set_ylabel("before", fontsize=10)
    axes[1, 0].set_ylabel("after", fontsize=10)
    fig.suptitle(title, fontsize=12)
    fig.tight_layout()
    if path:
        fig.savefig(path, facecolor="white")
    plt.show()


show_pairs(member_x, member_rt, mse_m, "Members (training images)", OUT / "members.png")
show_pairs(nonmember_x, nonmember_rt, mse_n, "Non-members (CIFAR-10 test images)", OUT / "nonmembers.png")

## Mechanism: members are near fixed points

An overfit denoiser has learned a score field whose attractors are its
training images. A member, noised a little and denoised, falls straight
back into its own basin — it is (nearly) a **fixed point** of the
round-trip. A non-member gets pulled toward the nearest memorized training
mode instead, so it moves much further. The MSE histograms below are that
gap; the attack is just a threshold on it.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.4), dpi=100)
bins = np.logspace(np.log10(min(mse_m.min(), mse_n.min())), np.log10(max(mse_m.max(), mse_n.max())), 40)
ax.hist(mse_m, bins=bins, color=TEAL, alpha=0.7, label="members")
ax.hist(mse_n, bins=bins, color=RED, alpha=0.6, label="non-members")
ax.set_xscale("log"); ax.set_xlabel("pixel-MSE after one round-trip"); ax.set_ylabel("images")
ax.legend(frameon=False)
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
fig.tight_layout(); fig.savefig(OUT / "mse_hist.png", facecolor="white"); plt.show()

**Iterating does not help.** A natural idea is to run the round-trip
several times and let members and non-members drift further apart. In the
lecturer's research run this is a confirmed *negative* result: the attack
is strongest after a single round-trip (k = 1) and gets weaker as the loop
is repeated — the non-members also settle onto memorized modes and stop
moving, so the gap closes. That is why this demo is one round-trip, not a
loop. Try it yourself in exercise 2.

## Optional: the memorization gate at larger n

The signal above exists because 500 images are few enough for this model to
memorize. Train the same model on `N_LARGE = 5000` images and the gap mostly
closes (AUROC ≈ 0.52 in our test run, with 4000 steps per model). Set `RUN_LARGE = True` in the
config cell (or here) to run it.

**Time warning:** this trains a second model with the **same number of
optimizer steps** (`TRAIN_STEPS`), i.e. about the same time again
(roughly 7–13 minutes on a T4) — but that means each image is seen 10× less often.
This is a cheaper, not identical, version of the comparison on the slide:
the research run trained every model for the same 400 *epochs*, so its
n = 10 000 model saw each image just as often as the n = 500 one and still
did not memorize it. To match that here, set the large run's steps to
`TRAIN_STEPS * N_LARGE // N_MEMBER` (10× longer, over an hour on a T4).
Members are scored on the first `PER_GROUP` images of the same permutation,
which are also members of the small run; the non-members are the same.

In [ ]:
results = {f"n={N_MEMBER}": res_small}
if RUN_LARGE:
    large_idx = member_indices(N_LARGE)
    large_train = torch.stack([ds_train[i][0] for i in large_idx])
    t_large = time.time()
    unet_large = train_ddpm(large_train, TRAIN_STEPS)
    print(f"trained in {time.time() - t_large:.0f}s")
    res_large, *_ = attack(unet_large, large_train[:PER_GROUP], f"n_member={N_LARGE}")
    results[f"n={N_LARGE}"] = res_large
    plot_roc_loglog(results, OUT / "roc_loglog.png")
else:
    print("skipped (RUN_LARGE = False)")

## Save results.json

**Single seed.** Every number here comes from one training seed and one
noise draw per image (no averaging over SDEdit replicas). The AUROC is
fairly stable; the TPR@1 %FPR figure is not — with 500 non-members it rests
on the 5 lowest-scoring of them. Do not quote one run's number without
saying "n_member = 500, seed 0, single replica".

In [ ]:
summary = {
    "config": {"n_member": N_MEMBER, "per_group": PER_GROUP, "train_steps": TRAIN_STEPS,
               "batch": BATCH, "lr": LR, "t_edit": T_EDIT, "ddim_steps": DDIM_STEPS,
               "seed": SEED, "device": DEVICE},
    "results": {k: {m: round(v, 4) for m, v in r.items() if m not in ("fpr", "tpr")}
                for k, r in results.items()},
}
(OUT / "results.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary["results"], indent=2))

## Exercises

1. **Training budget.** Halve and double `TRAIN_STEPS`. How does AUROC at
   n_member = 500 move? The research run's 400 epochs correspond to
   about `TRAIN_STEPS = 3100` at batch 64 (the default 5000 is ~640
   epochs) — but with a different model and schedule, so the numbers still
   will not match exactly. One test run of this notebook's model (Apple M4,
   seed 0) gave AUROC 0.54 / 0.63 / 0.81 / 0.97 / 0.99 after 1000 / 2000 /
   3000 / 4000 / 5000 steps — memorization switches on fairly suddenly, and
   just below the default the result moves between runs (a second run gave
   0.935 at 4000 steps). That is why the default is 5000, not 4000.
2. **Iterating does not help.** Feed the round-trip's output back in
   (`x_k = roundtrip(unet, x_{k-1}, seed=k)`) for k = 1…5 and compute AUROC of
   `mse(x_k, x_0)` at each k. Does the gap grow or shrink?
3. **`t_edit` is a knob with a sweet spot.** Try `t_edit` in 0.10, 0.25,
   0.50, 0.75. Too little noise and every image comes back unchanged; too much
   and every image is replaced. Where is the member/non-member gap largest?
4. **Connect to P3.** If the target model in your project overfits, your
   attack should show exactly this gate. What does that say about
   reporting MIA results without stating the training-set size and epochs?